In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="TE2gtVnw3YrYI4zH8Mcb")
project = rf.workspace("v-0mqey").project("traffic-imvd8")
version = project.version(8)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 35.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 76.5 MB/s eta 0:00:00:00:01
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.11.0.86
    Uninstalling opencv-python-headless-4.11.0.86:
      Successfully uninstalled opencv-python-headless-4.11.0.86
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-sto


Extracting Dataset Version Zip to traffic-8 in yolov8:: 100%|██████████| 11092/11092 [00:01<00:00, 7780.62it/s] 


In [2]:
import os
import shutil
import random
from collections import defaultdict

# ===============================
# Config
# ===============================
IMG_DIR = "/kaggle/working/traffic-8/train/images"      # original images
LBL_DIR = "/kaggle/working/traffic-8/train/labels"      # original YOLO labels
OUT_DIR = "/kaggle/working/dataset_splits"      # new split dataset

splits = {"train": 0.8, "valid": 0.1, "test": 0.1}

# Make output dirs
for split in splits:
    os.makedirs(os.path.join(OUT_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, split, "labels"), exist_ok=True)

# ===============================
# Group images by class
# ===============================
class_to_images = defaultdict(list)

for lbl_file in os.listdir(LBL_DIR):
    if not lbl_file.endswith(".txt"):
        continue

    lbl_path = os.path.join(LBL_DIR, lbl_file)
    with open(lbl_path, "r") as f:
        lines = f.readlines()

    if not lines:
        continue

    # Take first class ID (assuming single-class per image)
    cls_id = int(lines[0].split()[0])

    if cls_id == 0:
        cname = "green"
    elif cls_id == 1:
        cname = "red"
    elif cls_id == 2:
        cname = "yellow"
    else:
        continue

    img_file = lbl_file.replace(".txt", ".jpg")  # change if your images are .png
    img_path = os.path.join(IMG_DIR, img_file)

    if os.path.exists(img_path):
        class_to_images[cname].append((img_path, lbl_path))

# ===============================
# Stratified Split
# ===============================
for cname, items in class_to_images.items():
    random.shuffle(items)
    n = len(items)

    n_train = int(n * splits["train"])
    n_valid = int(n * splits["valid"])
    n_test  = n - n_train - n_valid

    split_data = {
        "train": items[:n_train],
        "valid": items[n_train:n_train+n_valid],
        "test":  items[n_train+n_valid:]
    }

    # Copy files
    for split, pairs in split_data.items():
        for img_path, lbl_path in pairs:
            shutil.copy(img_path, os.path.join(OUT_DIR, split, "images", os.path.basename(img_path)))
            shutil.copy(lbl_path, os.path.join(OUT_DIR, split, "labels", os.path.basename(lbl_path)))

    print(f"{cname}: Train={len(split_data['train'])}, Valid={len(split_data['valid'])}, Test={len(split_data['test'])}")

print("\n✅ Stratified 80/10/10 split completed at:", OUT_DIR)

# ===============================
# Write new data.yaml
# ===============================
yaml_content = """names:
- green
- red
- yellow
nc: 3
train: {}/train/images
val: {}/valid/images
test: {}/test/images
""".format(OUT_DIR, OUT_DIR, OUT_DIR)

with open(os.path.join(OUT_DIR, "data.yaml"), "w") as f:
    f.write(yaml_content)

print("✅ data.yaml saved at:", os.path.join(OUT_DIR, "data.yaml"))

green: Train=1468, Valid=183, Test=184
red: Train=1424, Valid=178, Test=178
yellow: Train=1454, Valid=181, Test=183

✅ Stratified 80/10/10 split completed at: /kaggle/working/dataset_splits
✅ data.yaml saved at: /kaggle/working/dataset_splits/data.yaml


In [ ]:
# =============================
# YOLOv8 Advanced Training for Higher mAP
# =============================
!pip install --upgrade ultralytics albumentations -q

from ultralytics import YOLO

# -------------------------
# Config
# -------------------------
DATA_YAML    = "/kaggle/working/dataset_splits/data.yaml"
BASE_WEIGHTS = "yolov8m.pt"   # Medium model (better accuracy than yolov8s)
BATCH        = 8              # Increase batch size if VRAM allows
WORKERS      = 2
CACHE        = True           # Enable dataset caching
IMG_SIZE     = 640
EPOCHS       = 150            # Longer training (papers use 100-300)
PATIENCE     = 30             # Allow more epochs before early stop

# -------------------------
# Load model
# -------------------------
model = YOLO(BASE_WEIGHTS)

# -------------------------
# Train with stronger augmentations + tuned hyperparams
# -------------------------
model.train(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    device=0,
    amp=True,
    workers=WORKERS,
    cache=CACHE,
    optimizer="AdamW",       # Stable optimizer
    lr0=0.002,               # Slightly higher initial LR
    lrf=0.01,                # Cosine LR final factor
    cos_lr=True,             # Cosine LR decay
    warmup_epochs=3,         # Longer warmup for stability
    weight_decay=0.0005,     # Regularization
    patience=PATIENCE,
    mixup=0.2,               # Stronger augmentation
    copy_paste=0.3,          # Copy-Paste augmentation
    mosaic=1.0,              # Keep mosaic active longer
    hsv_h=0.015,             # Color augmentation
    hsv_s=0.7,
    hsv_v=0.4,
    perspective=0.0005,      # Geometric distortion
    flipud=0.5,              # Vertical flips
    fliplr=0.5,              # Horizontal flips
    name="advanced_yolov8"
)

# -------------------------
# Validate
# -------------------------
metrics = model.val()
print(metrics)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.7 MB/s eta 0:00:00:00:0100:01


In [5]:
metrics = model.val()  # Validation returns a Results object

# The Results object has properties like:
# metrics.box: contains metrics for bounding boxes
# metrics.box.precision - not directly accessible; instead, use metrics.box.map or metrics.key to find available.
# Convert Results to dictionary to inspect keys and values for metrics
# metrics_dict = metrics.metrics or metrics.box or metrics.value or {} # This line caused the error

print("\n=== YOLOv8 Evaluation Metrics ===")

# Example printout:
# You may need to adjust keys based on what your version returns

print(f"mAP@0.5:0.95  : {metrics.box.map:.4f}" if hasattr(metrics.box, "map") else "mAP@0.5:0.95  : N/A")
print(f"mAP@0.5       : {metrics.box.map50:.4f}" if hasattr(metrics.box, "map50") else "mAP@0.5       : N/A")

# Precision, Recall, F1 may be available as ~metrics.box.pr, metrics.box.recall, metrics.box.f1 or within dictionary keys

# If not available, access as:
# The error message indicates that 'pr', 're', and 'f1' might not be direct attributes.
# Let's try accessing them through the results_dict or mean_results if available.
# Based on the traceback, mean_results is a method that returns precision, recall, mAP50, and mAP50-95.
# We can also access the metrics directly from the 'box' attribute as shown in the traceback.

# Accessing precision, recall, and F1 from the 'box' attribute if available
if hasattr(metrics.box, "p") and hasattr(metrics.box, "r"):
    # Access the mean values from the arrays
    print(f"Precision    : {metrics.box.p.mean():.4f}")
    print(f"Recall       : {metrics.box.r.mean():.4f}")
    # Calculate F1 score from the mean precision and recall
    p_mean = metrics.box.p.mean()
    r_mean = metrics.box.r.mean()
    f1 = 2 * (p_mean * r_mean) / (p_mean + r_mean) if (p_mean + r_mean) > 0 else 0
    print(f"F1 Score     : {f1:.4f}")
else:
    # If not available as direct attributes, we can try accessing the mean_results
    try:
        p, r, map50, map = metrics.mean_results()
        print(f"Precision    : {p:.4f}")
        print(f"Recall       : {r:.4f}")
        # F1 score needs to be calculated manually if not directly available
        f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        print(f"F1 Score     : {f1:.4f}")
    except Exception as e:
        print("Precision    : N/A")
        print("Recall       : N/A")
        print("F1 Score     : N/A")
        print(f"Could not calculate P, R, F1: {e}")

Ultralytics 8.3.196 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1433.8±664.3 MB/s, size: 55.5 KB)
val: Scanning /kaggle/working/dataset_splits/valid/labels.cache... 523 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 523/523 1.1Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 131/131 17.2it/s 7.6s0.1s


invalid value encountered in less
invalid value encountered in less


                   all        523       1243      0.921      0.843      0.919      0.596
                 green        195        393      0.914      0.827      0.911      0.593
                   red        209        457       0.94      0.893       0.96      0.677
                yellow        204        393      0.909       0.81      0.887      0.517
Speed: 0.5ms preprocess, 10.7ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /kaggle/working/runs/detect/train23

=== YOLOv8 Evaluation Metrics ===
mAP@0.5:0.95  : 0.5958
mAP@0.5       : 0.9192
Precision    : 0.9210
Recall       : 0.8431
F1 Score     : 0.8803
